# Data Collection

## Area of Interest
Our initial step is to define an AOI(Area Of Interest), eg. the land area we want to perform the analysis on. On the Copernicus Browser website (https://browser.dataspace.copernicus.eu) I located the area around Ljusdal, Sweden where the fire took form and, using their website tools, drew a bounding box around the afflicted area. The website returns the box as a WGS84 polygon string.

In [1]:
# Area of interest as the WGS84 polygon string
aoi = {"type":"Polygon","coordinates":[[[15.112381,62.07592],[15.621872,62.07592],[15.621872,61.87687],[15.112381,61.87687],[15.112381,62.07592]]]}

Next we will have to send a request to the Copernicus.eu ODATA archive to retrieve two(2) Sentinel-2 L2A images on my AOI, one before the fire, and one after it has settled.

## Request Authentication

The request requires an OAuth token for authentication, which in turn requires an account at their website. Credentials are read from a `.env` file (see `.env.example`) rather than hardcoded here.

NOTE: The token has a 10 minute lifespan, if you at any point get a 401 error, run the get_token cell again to refresh it.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env in the working directory

USERNAME: str = os.environ["CDSE_USERNAME"]
PASSWORD: str = os.environ["CDSE_PASSWORD"]

In [3]:
import requests

def get_token(username: str, password: str):
    r = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={
            "client_id": "cdse-public",
            "username": username,
            "password": password,
            "grant_type": "password",
        }
    )
    r.raise_for_status()
    return r.json()["access_token"]

token = get_token(USERNAME, PASSWORD)

## Search Viable Scenes

Below are the functions necessary to send a request to Copernicus ODATA and search for viable scenes that intersect with our AIO, with a max cloud cover percentage parameter to find clear-sky scenes.

I will be requesting a S2MI2A scene which has L2A data. Compared to L1C data, L2A is a post-processed version of original L1C data that corrects for atmospheric effects such as scattering and absorption to produce surface reflactance values. I chose L2A over L1C because I will be comparing band ratios across two different scenes and dates, so I want consistent surface reflectance.

In [4]:
# Extract only the actual polygon coordinates which are in a WGS84 format, into a WKT format for the request
def polygon_to_wkt(geojson_polygon):
    coords = geojson_polygon["coordinates"][0]
    coord_str = ", ".join(f"{lon} {lat}" for lon, lat in coords)
    return f"POLYGON(({coord_str}))"

def search_scenes(polygon, date_start: str, date_end: str , cloud_cover_max: int=10):
    wkt = polygon_to_wkt(polygon)
    url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
    params = {
        "$filter": (
            f"Collection/Name eq 'SENTINEL-2' and "
            f"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' "
            f"and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A') and "
            f"Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' "
            f"and att/OData.CSC.DoubleAttribute/Value le {cloud_cover_max}) and "
            f"ContentDate/Start gt {date_start}T00:00:00.000Z and "
            f"ContentDate/Start lt {date_end}T00:00:00.000Z and "
            f"OData.CSC.Intersects(area=geography'SRID=4326;{wkt}')"
        ),
        "$orderby": "ContentDate/Start",
        "$top": 10,
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    return r.json()["value"]

I will be using the "search_scenes" function to search for two scenes specifically: one before the fire, and one after. Since the fire happened around mid-juli, I have specified the dates as such below. I choose these dates because they cover approximately 1 month before the fire broke out, and until 2 months after. 

I will be increasing the "cloud_cover_max" for the post-fire scene because cloud coverage increases during the late-summer / autumn months in Sweden.

In [5]:
scenes_before = search_scenes(aoi, date_start="2018-06-01", date_end="2018-07-10", cloud_cover_max=10)
scenes_after  = search_scenes(aoi, date_start="2018-09-01", date_end="2018-09-30", cloud_cover_max=35)

print("=== PRE-FIRE ===")
for s in scenes_before:
    print(s["Name"], s["Id"])

print("\n=== POST-FIRE ===")
for s in scenes_after:
    print(s["Name"], s["Id"])

=== PRE-FIRE ===
S2B_MSIL2A_20180602T104019_N0500_R008_T33VWJ_20230828T001022.SAFE 7b8872f9-4d24-48ae-824b-1c5e0e6dbec6
S2B_MSIL2A_20180602T104019_N0500_R008_T33VVJ_20230828T001022.SAFE 8086f219-86ac-4e73-bbd2-11a7f3d79241
S2A_MSIL2A_20180604T103021_N0500_R108_T33VWJ_20230723T023511.SAFE 7cfb8053-d538-4906-af45-b06ab137fe29
S2A_MSIL2A_20180604T103021_N0500_R108_T33VVJ_20230723T023511.SAFE c0a27cdf-d4f6-4591-b690-20f8e06bb319
S2B_MSIL2A_20180612T104019_N0500_R008_T33VWJ_20230716T012327.SAFE 3d168992-0141-4702-b9f5-c3a68bd49f6f
S2B_MSIL2A_20180612T104019_N0500_R008_T33VVJ_20230716T012327.SAFE a4b2108e-bf29-4c89-94dc-b29537b53355
S2A_MSIL2A_20180701T102021_N0500_R065_T33VWJ_20230712T162421.SAFE 9c39af23-6963-431e-bfac-00259beeff86
S2A_MSIL2A_20180701T102021_N0500_R065_T33VVJ_20230712T162421.SAFE b080a71e-7de8-48f9-8b1f-411999a1a325
S2B_MSIL2A_20180702T104019_N0500_R008_T33VWJ_20230712T235932.SAFE 42cde111-9220-46d8-a110-6637ad871d3e
S2B_MSIL2A_20180702T104019_N0500_R008_T33VVJ_20230712T23

The scenes listed each have a name, which we can use to search for a specific scene and compare them directly in the browser before we download them.

## Scene Download

After viewing the scenes online, I have selected these dates (2018-07-02 & 2018-09-02) as viable scenes for the analysis based on these metrics:
1. Low or no cloud coverage over the fire area.
2. The smoke from the fire has cleared
3. The burn signal has stabilized in the spectral response
4. Up to 1 month pre-fire, and up to 2 months post-fire, to allow the above and to not stray to far into the future where vegetation might recover.

In [6]:
import os

def download_product(product_id, out_path, token):
    url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"
    headers = {"Authorization": f"Bearer {token}"}
    r = requests.get(url, headers=headers, stream=True)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    downloaded = 0
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                print(f"\r{downloaded / total * 100:.1f}%", end="", flush=True)
    print(f"\nSaved to {out_path}")

os.makedirs("data/raw", exist_ok=True)

# Here are the scenes I picked, indexed from the previous lists
scene_pre  = scenes_before[8]
scene_post = scenes_after[0]

# Download the actual scenes
print(f"Downloading pre-fire:  {scene_pre["Name"]}")
download_product(scene_pre["Id"],  f"data/raw/{scene_pre["Name"]}.zip",  token)

print(f"Downloading post-fire: {scene_post["Name"]}")
download_product(scene_post["Id"], f"data/raw/{scene_post["Name"]}.zip", token)

100.0%
Saved to data/raw/S2B_MSIL2A_20180702T104019_N0500_R008_T33VWJ_20230712T235932.SAFE.zip
100.0%
Saved to data/raw/S2A_MSIL2A_20180902T103021_N0500_R108_T33VWJ_20230710T085832.SAFE.zip


## Extract the scene folders

Since the downloaded scenes are compressed, I will unzip them to be able to work with the data.

In [ ]:
from zipfile import ZipFile

# Extract pre-fire scenes
with ZipFile(f"data/raw/{scene_pre["Name"]}.zip", "r") as z:
    z.extractall(path="data/raw")

# Extract post-fire scenes
with ZipFile(f"data/raw/{scene_post["Name"]}.zip", "r") as z:
    z.extractall(path="data/raw")

Now with our scenes downloaded and extracted, we can begin our preprocessing. This is shown in notebook number 2; "02_preprocessing.ipynb"